In [1]:
import numpy as np

np.random.seed(0)

# Dummy dataset (XOR-like)
X = np.array([[0,0],[0,1],[1,0],[1,1]])
y = np.array([[0],[1],[1],[0]])

# Initialize weights
W1 = np.random.randn(2, 3)
b1 = np.zeros((1, 3))
W2 = np.random.randn(3, 1)
b2 = np.zeros((1, 1))

lr = 0.1

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_deriv(x):
    return x * (1 - x)

for epoch in range(3):
    print(f"\n=== Epoch {epoch} ===")

    # Forward pass
    z1 = X @ W1 + b1
    a1 = sigmoid(z1)
    z2 = a1 @ W2 + b2
    y_pred = sigmoid(z2)

    # Loss
    loss = np.mean((y - y_pred) ** 2)
    print("Loss:", loss)

    # Backprop
    dL_dy = 2 * (y_pred - y)
    dL_dz2 = dL_dy * sigmoid_deriv(y_pred)

    dL_dW2 = a1.T @ dL_dz2
    dL_db2 = np.sum(dL_dz2, axis=0)

    dL_da1 = dL_dz2 @ W2.T
    dL_dz1 = dL_da1 * sigmoid_deriv(a1)

    dL_dW1 = X.T @ dL_dz1
    dL_db1 = np.sum(dL_dz1, axis=0)

    print("Before update W2:\n", W2)

    # Update weights
    W2 -= lr * dL_dW2
    b2 -= lr * dL_db2
    W1 -= lr * dL_dW1
    b1 -= lr * dL_db1

    print("After update W2:\n", W2)

    print("Predictions:\n", y_pred)


=== Epoch 0 ===
Loss: 0.2579005381593666
Before update W2:
 [[ 0.95008842]
 [-0.15135721]
 [-0.10321885]]
After update W2:
 [[ 0.93335859]
 [-0.17035307]
 [-0.11693821]]
Predictions:
 [[0.58607335]
 [0.66805998]
 [0.6559865 ]
 [0.6779312 ]]

=== Epoch 1 ===
Loss: 0.25401824300848364
Before update W2:
 [[ 0.93335859]
 [-0.17035307]
 [-0.11693821]]
After update W2:
 [[ 0.918505  ]
 [-0.18777566]
 [-0.12952095]]
Predictions:
 [[0.57302709]
 [0.65384384]
 [0.64144925]
 [0.66281989]]

=== Epoch 2 ===
Loss: 0.2507936350037066
Before update W2:
 [[ 0.918505  ]
 [-0.18777566]
 [-0.12952095]]
After update W2:
 [[ 0.90548143]
 [-0.20365166]
 [-0.14099482]]
Predictions:
 [[0.56103127]
 [0.64065744]
 [0.62798169]
 [0.64876325]]


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(0)

# Dummy image data (batch=1, channel=1, 4x4)
X = torch.randn(1, 1, 4, 4)
y = torch.tensor([[1.0]])

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 1, kernel_size=2)
        self.fc = nn.Linear(9, 1)

    def forward(self, x):
        x = self.conv(x)
        x = torch.relu(x)
        x = x.view(1, -1)
        x = self.fc(x)
        return torch.sigmoid(x)

model = SimpleCNN()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

for epoch in range(3):
    print(f"\n=== Epoch {epoch} ===")

    optimizer.zero_grad()

    output = model(X)
    loss = criterion(output, y)

    print("Loss:", loss.item())
    print("Prediction:", output.detach().numpy())

    # Before update
    print("Conv weights BEFORE:\n", model.conv.weight.data)

    loss.backward()

    # Gradients
    print("Conv weight gradients:\n", model.conv.weight.grad)

    optimizer.step()

    # After update
    print("Conv weights AFTER:\n", model.conv.weight.data)


=== Epoch 0 ===
Loss: 0.25199249386787415
Prediction: [[0.49801147]]
Conv weights BEFORE:
 tensor([[[[ 0.1977,  0.3000],
          [-0.3390, -0.2177]]]])
Conv weight gradients:
 tensor([[[[0.0284, 0.0458],
          [0.0590, 0.1071]]]])
Conv weights AFTER:
 tensor([[[[ 0.1948,  0.2954],
          [-0.3449, -0.2284]]]])

=== Epoch 1 ===
Loss: 0.23642049729824066
Prediction: [[0.5137691]]
Conv weights BEFORE:
 tensor([[[[ 0.1948,  0.2954],
          [-0.3449, -0.2284]]]])
Conv weight gradients:
 tensor([[[[0.0222, 0.0439],
          [0.0611, 0.1153]]]])
Conv weights AFTER:
 tensor([[[[ 0.1926,  0.2910],
          [-0.3510, -0.2400]]]])

=== Epoch 2 ===
Loss: 0.22112806141376495
Prediction: [[0.52975744]]
Conv weights BEFORE:
 tensor([[[[ 0.1926,  0.2910],
          [-0.3510, -0.2400]]]])
Conv weight gradients:
 tensor([[[[0.0164, 0.0419],
          [0.0627, 0.1226]]]])
Conv weights AFTER:
 tensor([[[[ 0.1910,  0.2868],
          [-0.3572, -0.2522]]]])


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(0)

# Dummy sequence (seq_len=3, input_size=1)
X = torch.tensor([[[0.1],[0.2],[0.3]]])
y = torch.tensor([[1.0]])

class SimpleRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(input_size=1, hidden_size=2, batch_first=True)
        self.fc = nn.Linear(2, 1)

    def forward(self, x):
        out, h = self.rnn(x)
        out = self.fc(out[:, -1, :])  # last time step
        return torch.sigmoid(out)

model = SimpleRNN()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

for epoch in range(3):
    print(f"\n=== Epoch {epoch} ===")

    optimizer.zero_grad()

    output = model(X)
    loss = criterion(output, y)

    print("Loss:", loss.item())
    print("Prediction:", output.detach().numpy())

    # Before update
    print("RNN weights BEFORE:\n", model.rnn.weight_ih_l0.data)

    loss.backward()

    # Gradients
    print("RNN gradients:\n", model.rnn.weight_ih_l0.grad)

    optimizer.step()

    # After update
    print("RNN weights AFTER:\n", model.rnn.weight_ih_l0.data)


=== Epoch 0 ===
Loss: 0.4561288058757782
Prediction: [[0.32462692]]
RNN weights BEFORE:
 tensor([[-0.0053],
        [ 0.3793]])
RNN gradients:
 tensor([[0.0133],
        [0.0028]])
RNN weights AFTER:
 tensor([[-0.0066],
        [ 0.3790]])

=== Epoch 1 ===
Loss: 0.44054940342903137
Prediction: [[0.33626106]]
RNN weights BEFORE:
 tensor([[-0.0066],
        [ 0.3790]])
RNN gradients:
 tensor([[0.0139],
        [0.0019]])
RNN weights AFTER:
 tensor([[-0.0080],
        [ 0.3788]])

=== Epoch 2 ===
Loss: 0.4249003231525421
Prediction: [[0.3481562]]
RNN weights BEFORE:
 tensor([[-0.0080],
        [ 0.3788]])
RNN gradients:
 tensor([[0.0144],
        [0.0011]])
RNN weights AFTER:
 tensor([[-0.0095],
        [ 0.3787]])
